In [ ]:
from langchain_text_splitters import SomeTextSplitter

# === 표준 호출 형태 ===
splitter = SomeTextSplitter(chunk_size=..., chunk_overlap=...)

split_docs = splitter.split_documents(docs)              # Document 리스트 → Document 리스트
chunks    = splitter.split_text(long_text)               # 문자열 → 문자열 리스트
new_docs  = splitter.create_documents(texts, metadatas)  # 원시 텍스트 + metadata → Document 리스트

---

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# === 객체 생성 ===
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                                # 한 조각 최대 글자 수
    chunk_overlap=50,                              # 인접 조각 겹침
    separators=["\n\n", "\n", " ", ""],            # 우선순위 구분자 (기본값)
    add_start_index=False,                         # True면 metadata에 시작 위치 추가
)

# === Document 리스트 분할 ===
split_docs = splitter.split_documents(docs)

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# === 객체 생성 ===
splitter = CharacterTextSplitter(
    separator="\n\n",         # 분할 구분자 (기본값)
    chunk_size=500,
    chunk_overlap=50,
)

# === Document 리스트 분할 ===
split_docs = splitter.split_documents(docs)

In [ ]:
from langchain_text_splitters import TokenTextSplitter

# === 객체 생성 ===
splitter = TokenTextSplitter(
    chunk_size=500,                  # 토큰 수 기준
    chunk_overlap=50,                # 토큰 단위 겹침
    encoding_name="cl100k_base",     # GPT-4 / GPT-3.5 등 OpenAI 모델용
    # 또는 model_name="gpt-4o-mini"
)

# === Document 리스트 분할 ===
split_docs = splitter.split_documents(docs)

In [ ]:
# RecursiveCharacterTextSplitter + length_function 패턴
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,                                   # 토큰 수 기준
    chunk_overlap=10,
    length_function=lambda t: len(enc.encode(t)),    # 글자 수가 아니라 토큰 수로 길이 계산
    separators=["\n\n", "\n", ". ", " ", ""],
)

split_docs = splitter.split_documents(docs)

for i, d in enumerate(split_docs, 1):
    n_tokens = len(enc.encode(d.page_content)) # length_function
    print(f"[{i}] (tokens={n_tokens}) {d.page_content!r}\n")

In [ ]:
# Gemini 기준 토큰 수 계산하기
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# === Gemini ChatModel 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key="YOUR_API_KEY",
)

# === 샘플 문서 준비 ===
text = (
    "Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다. "
    "어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다. "
    "5단 분석법은 일반 명사, 고유 명사, 사용 이유, 사용 방법, 다른 기술과의 비교 다섯 단계로 구성됩니다. "
    "어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."
)

docs = [Document(page_content=text, metadata={"source": "adapterz_intro.txt"})]

# === Gemini 모델 기준 토큰 수로 길이 계산 ===
splitter_gemini = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=lambda t: model.get_num_tokens(t),
    separators=["\n\n", "\n", ". ", " ", ""],
)

# === 문서 분할 ===
split_docs = splitter_gemini.split_documents(docs)

print(f"분할 결과: {len(split_docs)}개\n")
for i, d in enumerate(split_docs, 1):
    n_tokens = model.get_num_tokens(d.page_content)
    print(f"[{i}] (gemini_tokens={n_tokens}, chars={len(d.page_content)})")
    print(f"    {d.page_content!r}")
    print(f"    metadata={d.metadata}\n")